<style>
.jp-Notebook {max-width: 1180px; margin: auto; padding: 24px;}
.jp-InputPrompt, .jp-OutputPrompt {display: none !important;}
.jp-RenderedHTMLCommon p, .jp-RenderedHTMLCommon li {line-height: 1.65;}
.jp-RenderedHTMLCommon h2 {margin-top: 2.2em; padding-top: .5em; border-top: 1px solid #e2e5e9;}
.guide-nav {display:flex; flex-wrap:wrap; gap:10px 20px; padding:14px 0; border-bottom:1px solid #e2e5e9;}
.guide-nav a {font-weight:600;}
</style>
<a id="start"></a>
# Higgs diffusion: heuristic models (IC and LT)
**Aaron Liu & Matthew Galvez · CS 4352 · baseline results and implementation guide**

This report records the heuristic experiment: IC, LT, and simple controls. See the [experiment index](../../README.md) for the separate GNN, prompted-agent, and synthesis experiments. You can read it without installing anything. All plots and results are embedded; the notebook and Python files are there when you want to change something.

<div class="guide-nav"><a href="#data">Dataset & graph</a><a href="#mechanics">IC & LT</a><a href="#results">Results</a><a href="#gnn">Build the GNN</a><a href="#agents">Build the agents</a><a href="#working">Run & navigate</a></div>

**Question:** how well can simple network models explain how quickly people begin retweeting about the Higgs announcement? First establish IC/LT baselines; next learn an adoption rule with a small GNN; finally explore synthetic prompted agents. **Only the baseline stage is implemented.**

**How to read:** scan the graphs first. Read the dataset and model sections to understand their assumptions. Before coding, read the GNN/agent sections and the compact data contract at the end.

The default experiment uses 5,000 social-network users. It learns settings from the beginning of the week and evaluates the later continuation. A good fit early in the week does not guarantee a good forecast of the announcement surge.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
from pathlib import Path
import sys
# Find the shared project root from any directory inside the repository.
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents)
             if (p / "pyproject.toml").is_file() and (p / "diffusion").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the project repository")
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import hashlib
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Markdown
from diffusion.data import load_activity, sample_social, first_retweet_hours, cumulative_counts
from experiments.heuristic.experiment import run_experiment

# Start here: these are the only settings you need to change.
N_NODES = 5000
RANDOM_SEED = 42
SEED_HOUR = 24        # July 2 00:00 UTC: initialize from earlier first retweets
FIT_END = 72         # July 4 00:00 UTC: stop parameter selection
END_HOUR = 168       # July 8 00:00 UTC: end of observation
FIT_RUNS = 10        # per candidate setting
EVALUATION_RUNS = 50 # independent random draws, after selection

OUT = ROOT / "experiments" / "heuristic" / "results"
FIGURES = OUT / "figures"
EXPORT_ARTIFACTS = os.environ.get("HIGGS_EXPORT", "0") == "1"
if EXPORT_ARTIFACTS:
    FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 120,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": .18, "font.size": 10, "savefig.bbox": "tight"})
COLORS = {"Observed": "#1b263b", "IC": "#187e89", "LT": "#c87533",
          "No growth": "#9197a1", "Linear trend": "#785ba8"}
def save(name):
    plt.tight_layout()
    if EXPORT_ARTIFACTS:
        plt.savefig(FIGURES / f"{name}.png", dpi=160)
    plt.show()
def split_lines(ax):
    ax.axvline(FIT_END, color="#68707a", ls="--", lw=1)
    ax.axvspan(FIT_END, END_HOUR, color="#68707a", alpha=.05)
    ax.set_xlabel("Hours since July 1, 2012 00:00 UTC")

<a id="data"></a>
## 1. What is in these files?
The data track Twitter activity around the Higgs boson announcement during **July 1–7, 2012, in UTC**. The 2015 date on SNAP refers to an update. User IDs are anonymized and shared across the files. [Dataset documentation](https://snap.stanford.edu/data/higgs-twitter.html) · [Original study](https://www.nature.com/articles/srep02980)

- **Activity log:** `actor target unix_timestamp kind`. `RT` means retweet, `MT` mention, and `RE` reply. One row is an observed interaction, not necessarily a distinct tweet. This is our source of timing and labels.
- **Social network:** `follower followed`. A directed relationship, with no timestamp or weight. This supplies the possible routes for information flow.
- **Retweet network:** `actor target count`. Aggregates retweets over the entire week. We verify its weights against the activity log, but never use it as prediction topology: it contains later interactions we are trying to predict.

There is **no tweet text, tweet ID, biography, demographic profile, or original-post timestamp** in these files. We cannot reconstruct a specific message cascade or know when someone first heard the news.

### From interactions to an adoption label
**Adoption = an actor's first recorded RT in the week.** A later RT does not activate them again. Mentions/replies appear in activity charts but do not change this label. A retweet target is not automatically marked active. A user with no RT gets `infinity`, meaning unobserved through this window—not someone who will never adopt.

Example: `A B time RT` tells us A retweeted B at that time. It labels **A's participation**; it does not timestamp B's original post. This limitation matters when interpreting apparent missing exposure.

### From social edges to the simulation graph
1. Reverse each social edge: if **A follows B**, our influence edge is **B → A**.
2. Pick a root with random seed 42. Explore neighbors breadth first in randomized order, treating edges as undirected **only while selecting users**, until 5,000 users are selected. If a component runs out, continue from another root.
3. Keep every original directed influence edge whose endpoints are both selected. Remove self-loops; collapse duplicate edges. Remap sorted user IDs to indices 0 through N−1.
4. Store the graph as a sparse adjacency matrix. No activity labels choose the sample.

This neighborhood sample keeps many connections, but favors connected/high-degree neighborhoods. It is **not a representative random sample of Twitter**. Outside neighbors disappear, which changes exposure and LT's normalization. The social snapshot also is not guaranteed to predate the event, so this is retrospective evaluation with a fixed graph.


In [ ]:
events = load_activity("data/higgs-activity_time.txt.gz")
graph, node_ids, graph_info = sample_social("data/higgs-social_network.edgelist.gz", N_NODES, RANDOM_SEED)
first_hours = first_retweet_hours(events, node_ids)
retweet_edges = pd.read_csv("data/higgs-retweet_network.edgelist.gz", sep=r"\s+", names=["actor", "target", "weight"])
# Aggregate RT file is for a consistency check/description ONLY, never prediction.
rt_events = events.loc[events.kind == "RT"]
rt_counts = rt_events.groupby(["actor", "target"]).size().rename("events")
rt_weights = retweet_edges.set_index(["actor", "target"]).weight
rt_matches = rt_counts.sort_index().equals(rt_weights.sort_index().rename("events"))
display(Markdown(f"**{len(events):,} recorded interactions · {graph_info['full_nodes']:,} social users · "
    f"{graph_info['full_edges']:,} social edges**\n\n"
    f"Working sample: **{len(node_ids):,} users / {graph.nnz:,} edges**, with "
    f"**{np.isfinite(first_hours).sum():,} observed retweeters**. "
    f"RT aggregate weights match activity counts exactly: **{rt_matches}**."))
display(Markdown(f"Full RT network: **{len(retweet_edges):,} distinct directed pairs**, "
    f"**{int(retweet_edges.weight.sum()):,} retweets**. "
    f"Sample initialization: **{int((first_hours < SEED_HOUR).sum()):,} early retweeters** become seeds."))


<a id="mechanics"></a>
## How IC and LT work in this implementation
Both models hold a Boolean active/inactive state for every sampled user. Seeds start active; activity is permanent. Updates are **synchronous**: someone activated in this step can influence others only in the next step. No one enters from outside the seed-reachable network.

**Independent cascade (IC):** only newly active users attempt to activate each outgoing neighbor, once, with probability p. If several independent attempts reach v, its chance of at least one success is `1 − (1−p)^k` for k attempts. Failed edges are not retried. Stop when no new user activates.

**Linear threshold (LT):** draw one threshold per node from Uniform(0,1) at the start of each simulation; keep it fixed throughout that run. Each incoming edge has weight `scale / incoming_degree`, so the node's total incoming weight is at most 1. At each step, activate a node if `scale × fraction_of_active_incoming_neighbors ≥ threshold`, with at least one active neighbor. All active neighbors contribute, not only the newest ones. Zero-degree users cannot activate unless seeded.

**Mental model:** IC is a one-shot transmission attempt; LT is accumulating pressure that crosses a fixed personal threshold. The figure below is an illustration, not a result fitted to Higgs data. In the LT example, v has three incoming neighbors, scale=1, and threshold=0.6: one active neighbor is insufficient; two suffice.

Read [models.py](models.py) when you want the actual update loops. The implementation is about 50 lines; it returns active-user counts rather than storing full state histories.


In [ ]:
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(1, 2, figsize=(12, 3.7))
# Toy chain with p=1: A activates B one step later, then B activates C.
state = np.array([[1,0,0], [1,1,0], [1,1,1]])
axes[0].imshow(state, cmap=ListedColormap(["#edf0f3", "#187e89"]), vmin=0, vmax=1, aspect="auto")
axes[0].set(xticks=[0,1,2], xticklabels=["A", "B", "C"], yticks=[0,1,2],
            yticklabels=["0 (seed)", "1", "2"], xlabel="Toy chain: A → B → C", ylabel="Synchronous step",
            title="IC example: p = 1; one hop per step")
axes[0].grid(False)
for row in range(3):
    for col in range(3):
        axes[0].text(col,row,"Active" if state[row,col] else "Inactive",ha="center",va="center",
                     color="white" if state[row,col] else "#59616c")
axes[1].bar([1,2,3], [1/3,2/3,1], color=["#9197a1", "#c87533", "#c87533"])
axes[1].axhline(.6, ls="--", color="#1b263b", label="Fixed threshold = 0.6")
axes[1].set(xticks=[1,2,3], xlabel="Active incoming neighbors (out of 3)", ylabel="Total incoming influence",
            ylim=(0,1.12), title="LT example: scale = 1; threshold crossing")
axes[1].legend(loc="upper left")
axes[1].text(1,.38,"Not enough",ha="center")
axes[1].text(2,.72,"Activate v",ha="center")
save("00_model_mechanics")


<a id="results"></a>
## 2. What actually happened during the week?
The announcement occurred July 4. Activity volume counts interactions; adoption counts unique first-time retweeters. These measure different things. Sources for the next figures are the full activity log and the labeled social sample.

In [ ]:
hourly = pd.crosstab(np.floor(events.hour).astype(int), events.kind).reindex(range(168), fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for kind, color in zip(["RT", "MT", "RE"], ["#187e89", "#c87533", "#785ba8"]):
    axes[0].plot(hourly.index, hourly[kind], label=kind, color=color)
axes[0].axvline(72, color="#68707a", ls="--", label="July 4 starts")
axes[0].set(title="Full dataset: interactions per hour", xlabel="Hours since July 1 00:00 UTC", ylabel="Interactions / hour")
axes[0].legend()
counts = events.kind.value_counts().reindex(["RT", "MT", "RE"])
axes[1].bar(["Retweets", "Mentions", "Replies"], counts, color=["#187e89", "#c87533", "#785ba8"])
axes[1].set(title="Full dataset: interaction mix, July 1–7", ylabel="Recorded interactions")
for i, value in enumerate(counts): axes[1].text(i, value, f"{value:,}", ha="center", va="bottom")
save("01_activity")

hours = np.arange(169)
full_first = rt_events.groupby("actor").hour.min().to_numpy()
full_curve = cumulative_counts(full_first, hours)
sample_curve = cumulative_counts(first_hours, hours)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hours, sample_curve, color=COLORS["Observed"])
axes[0].set(title="Sample: cumulative first retweeters", ylabel="Users with an observed first RT")
# Normalize by final adopters solely for descriptive shape comparison, never fitting.
axes[1].plot(hours, full_curve / full_curve[-1], label="Full dataset", color="#68707a")
axes[1].plot(hours, sample_curve / max(sample_curve[-1], 1), label="Sample", color="#187e89")
axes[1].set(title="Is the sample's timing similar?", ylabel="Fraction of each group's final adopters")
axes[1].legend()
for ax in axes: split_lines(ax)
save("02_observed_adoption")

## 3. Inspect the network you are actually modeling
The small drawing is illustrative, not the full simulation graph. Sampling and omitting outside neighbors change network structure and LT normalization. Compare different sample seeds before making broad claims.

In [ ]:
in_degree = np.asarray(graph.sum(axis=0)).ravel()
out_degree = np.asarray(graph.sum(axis=1)).ravel()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for degree, label, color in [(in_degree, "Incoming influence sources", "#187e89"), (out_degree, "Outgoing followers", "#c87533")]:
    x = np.sort(degree)
    axes[0].step(x, np.arange(1, len(x)+1)/len(x), label=label, color=color)
axes[0].set(xscale="symlog", title="Sample: degree distribution", xlabel="Degree (symlog scale)", ylabel="Fraction of sample users ≤ degree")
axes[0].legend()
# At most 120 nodes and 40 layout iterations: avoid drawing 218k edges.
view_nodes = np.argsort(out_degree)[-120:]
g = nx.from_scipy_sparse_array(graph[view_nodes][:, view_nodes], create_using=nx.DiGraph)
pos = nx.spring_layout(g, seed=RANDOM_SEED, iterations=40)
nx.draw_networkx_edges(g, pos, ax=axes[1], arrows=False, alpha=.06, width=.5)
colors = np.where(np.isfinite(first_hours[view_nodes]), first_hours[view_nodes], 180)
points = nx.draw_networkx_nodes(g, pos, ax=axes[1], node_size=18, node_color=colors, cmap="viridis", vmin=0, vmax=180)
axes[1].set_title("120 highest-outdegree sample users")
axes[1].axis("off")
fig.colorbar(points, ax=axes[1], label="First RT hour (180 = not observed)", shrink=.8)
save("03_network")

## 4. How the comparison is made
**Initialize → fit → test.** At hour 24 (July 2 00:00 UTC), initialize both models by marking everyone who retweeted earlier as active and everyone else as inactive. These initial adopters are the seeds. Next, fit the settings: try different IC transmission probabilities or LT influence scales, together with different hours per spreading step, and choose the combination whose simulated cumulative adopter counts best match the observations at hours 25–72. Finally, test those chosen settings: keep them fixed, let simulations continue from the same hour-24 starting state, and compare their predictions with observations at hours 73–168. The later observations are used only to score predictions, never to choose settings or update simulated users. Hour 72 is July 4 00:00 UTC, before the announcement surge; this split asks whether early growth can predict the later burst.

**How to read the figures below:** the heatmaps show early fitting error (hours 25–72) for each settings combination. The cumulative-adoption chart shows both periods; the shaded region after hour 72 is the held-out test period. Section 5 expresses those same curves as spread speed, and its bar charts summarize held-out performance and final adoption error. The later real data appear in the plots for comparison, but were not used to fit the models.

For each model, search **8 parameter values × 4 step durations**, averaging 10 runs per setting. Select the smallest RMSE on the early cumulative-adopter curve. Run the selected setting 50 more times using fresh random seeds. **The process always starts from the hour-24 seeds and continues forward; it is not reset using observed test adopters.**

We fit one model step to **1, 3, 6, or 12 hours**. For elapsed time `h`, the reported state is step `floor(h / step_hours)`. All users active before hour 24 get a fresh initial chance to transmit; their historical attempts are unknown. This is a snapshot approximation.

**Metrics:** cumulative RMSE penalizes large errors more; MAE is mean absolute error in users; final error is predicted minus observed final size; spread speed is new adopters per hour. Hourly cumulative errors are correlated, so they are not independent samples for a significance test.

**Two controls:** no growth retains only the seeds; linear trend extrapolates the mean observed growth from hours 24–72, capped at N users. It uses no test observations.

Bands show **10th–90th percentiles across simulation runs** on this one graph with fixed selected settings. They are not confidence intervals and omit uncertainty from sampling or choosing parameters. A star on the heatmaps marks the selected setting; a boundary optimum is worth investigating.


In [ ]:
result = run_experiment(graph, first_hours, SEED_HOUR, FIT_END, END_HOUR,
                        FIT_RUNS, EVALUATION_RUNS, RANDOM_SEED)
display(result["best"].rename(columns={"parameter": "IC p / LT scale", "fit_rmse": "Fit RMSE (users)"}).round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model in zip(axes, ["IC", "LT"]):
    pivot = result["tuning"].query("model == @model").pivot(index="step_hours", columns="parameter", values="fit_rmse")
    im = ax.imshow(pivot, aspect="auto", cmap="viridis_r")
    ax.set(xticks=range(len(pivot.columns)), xticklabels=[f"{v:g}" for v in pivot.columns],
           yticks=range(len(pivot.index)), yticklabels=pivot.index,
           xlabel="Per-edge p" if model == "IC" else "Incoming-weight scale",
           ylabel="Hours per model step", title=f"{model}: early-period fit error")
    chosen = result["best"].query("model == @model").iloc[0]
    ax.scatter(list(pivot.columns).index(chosen.parameter), list(pivot.index).index(chosen.step_hours),
               marker="*", s=140, color="white", edgecolor="black", label="Selected")
    ax.legend(loc="upper right", fontsize=8)
    fig.colorbar(im, ax=ax, label="RMSE (users)")
save("04_parameter_sensitivity")

In [ ]:
t, observed = result["hours"], result["observed"]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t, observed, color=COLORS["Observed"], lw=2.5, label="Observed first retweeters", zorder=5)
for model, draws in result["predictions"].items():
    ax.plot(t, draws.mean(axis=0), label=model, color=COLORS[model], ls="--" if len(draws)==1 else "-")
    if len(draws) > 1:
        lo, hi = np.quantile(draws, [.1, .9], axis=0)
        ax.fill_between(t, lo, hi, color=COLORS[model], alpha=.15)
split_lines(ax)
ax.set(title="Cumulative adoption: early fit → held-out continuation", ylabel="Active users in the 5,000-user sample" if N_NODES == 5000 else "Active users in sample")
ax.legend(ncol=3)
ax.text(.99, .08, "Shaded region: held-out hours · bands: 10–90% of simulation runs", transform=ax.transAxes, ha="right", fontsize=9)
save("05_diffusion_forecast")

## 5. Measure spread speed, not just the final total
The next chart uses non-overlapping six-hour increments. It reveals when adoption accelerates and when a model stops spreading. Fitted step durations can cause visible bursts; those are model artifacts.

The linear-trend control extends the mean growth rate from hours 24–72, capped at the number of users. The no-growth control keeps only the initial seeds.

In [ ]:
six = np.arange(0, len(t), 6)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[six][1:], np.diff(observed[six])/6, label="Observed", color=COLORS["Observed"], lw=2)
for model in ["IC", "LT", "Linear trend"]:
    mean = result["predictions"][model].mean(axis=0)
    ax.plot(t[six][1:], np.diff(mean[six])/6, label=model, color=COLORS[model])
split_lines(ax)
ax.set(title="Spread speed: new first retweeters per hour (6-hour bins)", ylabel="New adopters / hour")
ax.legend()
save("06_spread_speed")

metrics = result["metrics"].set_index("model")
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, field, title, ylabel in zip(axes, ["test_rmse", "final_error", "test_speed"],
        ["Held-out cumulative RMSE", "Final adoption error", "Mean held-out spread speed"],
        ["Users (lower is better)", "Predicted − observed users", "New adopters / hour"]):
    ax.bar(metrics.index, metrics[field], color=[COLORS[m] for m in metrics.index])
    ax.set(title=title, ylabel=ylabel)
    ax.tick_params(axis="x", rotation=30)
    ax.axhline(0, color="#68707a", lw=.5)
axes[2].axhline(result["observed_test_speed"], color=COLORS["Observed"], ls="--", label="Observed")
axes[2].legend()
save("07_held_out_metrics")
display(result["metrics"].round(2))

## 6. Why might the models fail?
Reachability gives an absolute structural ceiling: without external arrivals, no seed-based model can activate a user without a directed path from a seed. A loose ceiling does not mean activation is likely.

The exposure diagnostic counts sample adopters who had **no incoming neighbor with an earlier observed first RT**. Such cases may reflect external news, missing neighbors, unobserved source tweets, or the weakness of our adoption proxy. They do not prove independent adoption or causal influence.

In [ ]:
reachable = result["reachable"]
early_neighbor = np.full(len(node_ids), np.inf)
src, dst = graph.nonzero()
np.minimum.at(early_neighbor, dst, first_hours[src])
later_adopters = np.isfinite(first_hours) & (first_hours >= SEED_HOUR)
no_prior = later_adopters & (early_neighbor >= first_hours)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ["Initial seeds", "Observed by week end", "Reachable ceiling"]
values = [result["seeds"].sum(), np.isfinite(first_hours).sum(), reachable.sum()]
axes[0].bar(labels, values, color=["#9197a1", "#1b263b", "#187e89"])
axes[0].set(title="Closed-model reachability in the sample", ylabel="Users", ylim=(0, N_NODES * 1.12))
for i, value in enumerate(values): axes[0].text(i, value, f"{value:,}", ha="center", va="bottom")
axes[1].bar(["Earlier neighbor RT", "No earlier neighbor RT"], [later_adopters.sum()-no_prior.sum(), no_prior.sum()], color=["#187e89", "#c87533"])
axes[1].set(title="Post-seed adopters: observed prior exposure", ylabel="Users")
save("08_reachability_exposure")
unreachable_adopters = int((np.isfinite(first_hours) & ~reachable).sum())
display(Markdown(f"**{unreachable_adopters:,}** observed adopters are unreachable from the seeds. "
    f"**{no_prior.sum():,} / {later_adopters.sum():,}** post-seed adopters have no earlier-retweeting incoming neighbor in this sample."))

## 7. Compare how quickly models reach the same adoption milestones
Milestones are percentages of **all sampled users**, not each model's own eventual total. This avoids declaring a tiny cascade 'fast' simply because it stops early. Model times below use the ensemble-mean curve; they are not mean per-run first-passage times. Missing points mean the milestone was not reached by hour 168.

In [ ]:
milestones = [.02, .10, .20, .50]
all_curves = {"Observed": observed, **{m: d.mean(axis=0) for m,d in result["predictions"].items()}}
passages = []
fig, ax = plt.subplots(figsize=(11, 4))
for j, (model, curve) in enumerate(all_curves.items()):
    for i, fraction in enumerate(milestones):
        hits = np.flatnonzero(curve >= fraction*N_NODES)
        hour = float(t[hits[0]]) if len(hits) else np.nan
        passages.append(dict(model=model, fraction_of_sample=fraction, first_hour=hour,
                             at_initialization=bool(len(hits) and hits[0] == 0)))
        y = i + (j-2)*.12
        if len(hits): ax.scatter(hour, y, color=COLORS[model], label=model if i==0 else None, s=35)
        else: ax.text(END_HOUR+2, y, "×", color=COLORS[model], va="center")
ax.set(yticks=range(4), yticklabels=[f"{p:.0%} ({int(p*N_NODES):,} users)" for p in milestones],
       xlabel="First hour reaching milestone (since July 1 00:00 UTC)", ylabel="Adoption milestone",
       title="Time to common adoption milestones · × = not reached", xlim=(SEED_HOUR-4, END_HOUR+12), ylim=(-.5, 3.5))
# Explicit handles keep the legend complete even if a model never reaches a milestone.
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0], marker="o", color=COLORS[m], ls="", label=m) for m in all_curves], ncol=3)
save("09_milestone_times")
if EXPORT_ARTIFACTS:
    pd.DataFrame(passages).to_csv(OUT / "milestones.csv", index=False)

## 8. What to take away from this run
The table and graphs above are the evidence; a single model ranking is not the conclusion. Look for whether errors come from the surge, long tail, or both. Compare the early-fit winner with the held-out winner, and compare each network model with the simple linear trend.

The summary below is computed from the current run. Exact settings and input fingerprints are included so this report remains self-contained even when optional export files are absent.


In [ ]:
manifest = dict(config=dict(n_nodes=N_NODES, random_seed=RANDOM_SEED, seed_hour=SEED_HOUR,
    fit_end=FIT_END, end_hour=END_HOUR, fit_runs=FIT_RUNS, evaluation_runs=EVALUATION_RUNS),
    graph={k: v for k, v in graph_info.items() if k != "signature"}, rt_aggregate_matches_activity=bool(rt_matches),
    input_sha256={p.name: hashlib.file_digest(p.open("rb"), "sha256").hexdigest() for p in Path("data").glob("*.gz")},
    source_sha256={name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in [
        "diffusion/data.py", "diffusion/evaluation.py", "experiments/heuristic/models.py",
        "experiments/heuristic/experiment.py", "experiments/heuristic/analysis.ipynb", "uv.lock"]},
    packages={p: importlib.metadata.version(p) for p in ["numpy", "pandas", "scipy", "matplotlib", "networkx"]})

if EXPORT_ARTIFACTS:
    result["metrics"].to_csv(OUT / "metrics.csv", index=False)
    result["tuning"].to_csv(OUT / "parameter_search.csv", index=False)
    result["best"].to_csv(OUT / "selected_parameters.csv", index=False)
    pd.DataFrame({"node_id": node_ids, "first_rt_hour": np.where(np.isfinite(first_hours), first_hours, np.nan),
                  "seed": result["seeds"], "reachable": reachable}).to_csv(OUT / "sample_nodes.csv", index=False)
    curve_table = pd.DataFrame({"hour": t, "observed": observed, "held_out": t > FIT_END})
    for model, draws in result["predictions"].items():
        for suffix, values in [("mean", draws.mean(axis=0)), ("p10", np.quantile(draws, .1, axis=0)), ("p90", np.quantile(draws, .9, axis=0))]:
            curve_table[f"{model}_{suffix}"] = values
    curve_table.to_csv(OUT / "curves.csv", index=False)
    np.savez_compressed(OUT / "simulation_draws.npz", **{m.replace(" ", "_"): d for m,d in result["predictions"].items()})
    (OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))

best_test = result["metrics"].sort_values("test_rmse").iloc[0]
display(Markdown(f"Lowest held-out RMSE in this run: **{best_test['model']}**, "
    f"**{best_test['test_rmse']:.1f} users**. This is one sample and one temporal split, not a general model ranking."))
early_winner = result["best"].sort_values("fit_rmse").iloc[0]
display(Markdown(f"Early-fit winner: **{early_winner['model']}** (fit RMSE {early_winner['fit_rmse']:.1f} users). "
    f"Observed final adoption: **{observed[-1]:,} users**. "
    + "; ".join(f"{m} mean final adoption: **{d.mean(axis=0)[-1]:,.0f}**" for m,d in result["predictions"].items() if m in ["IC", "LT"])
    + ". Repeat across graph samples before treating this ranking as stable."))
from IPython.display import HTML
from html import escape
display(HTML("<details><summary>Exact run settings, input hashes, and package versions</summary><pre>" + escape(json.dumps(manifest, indent=2)) + "</pre></details>"))


<a id="gnn"></a>
## 9. Next milestone: learn the adoption rule with a small GNN
**Keep:** the graph, user-index mapping, first-RT labels, chronological split, seed set, and all the curve-based metrics. **Replace:** the fixed IC/LT update rule with a learned probability of an inactive user adopting in the next hour. This is a proposed next step, not a trained model in this report.

### What a GNN adds
Each user has a short feature vector. A layer averages information from their incoming influence neighbors, combines it with their own features, and learns how to transform the result. One layer sees one-hop neighbors; two layers can see two-hop neighborhoods. This is the neighborhood-aggregation idea behind [GraphSAGE (Hamilton, Ying & Leskovec, 2017)](https://proceedings.neurips.cc/paper/2017/hash/5dd9db5e033da9c6fb5ba83c7a7ebea9-Abstract.html).

Start with **one mean-aggregation layer and a small hidden vector**, implemented in PyTorch using the existing sparse graph. A second graph framework is optional. Suggested features: current activity, log in/out degree, and elapsed time. Do not add anonymized ID as a numeric predictor. Keep the first network small enough to train on CPU with one or two threads.

### The exact learning task
At hour boundary `h`:

```python
active = first_hours < h
at_risk = ~active
adopts_next_hour = (first_hours >= h) & (first_hours < h + 1)
# Train the loss only on at_risk users.
```

A sigmoid output gives a probability for each still-inactive user. Use a binary classification loss on those users only. Most user-hours have no new adoption; report precision–recall behavior and probability calibration alongside aggregate curves. If weighting or subsampling negative labels, check that predicted probabilities remain calibrated before sampling rollouts.

Fit on early transitions only. A possible internal split is **h=24…47 for training**, **h=48…71 for validation**, and **h=72…167 for held-out one-step predictions**. A transition at h uses information available before h and labels before h+1. Select settings on validation, optionally refit through h=71, and keep the later labels untouched. Fit any feature normalization on training data only. These are time splits on the same graph, not a held-out-user generalization claim.

### Matrix direction: the detail that is easiest to get wrong
`graph[u, v] = 1` means u can influence v. Incoming messages therefore use **`graph.T @ features`**, divided rowwise by incoming degree for a mean. Using `graph @ features` aggregates from the opposite direction. Handle zero incoming degree with denominator `max(degree, 1)`; that user receives zero neighbor message but can still use their own features.

### One-step accuracy and a diffusion forecast are different experiments
- **One-step / teacher-forced:** use the observed state at each hour. This measures local prediction with frequent observations.
- **Autonomous rollout:** start with the hour-24 seed state, compute probabilities, sample new activations, keep active users active, and feed that simulated state into the next step. Average repeated trajectories and reuse the report's evaluation hours. Do not inject true test activations. This matches the existing continuation protocol.

A later-origin forecast beginning at hour 72 is also useful, but then **reinitialize IC and LT at the same origin** before comparing. Do not compare a GNN repeatedly refreshed with real observations to baselines left running from hour 24.

Use two small controls: logistic prediction from the same base features, and optionally a logistic model with the fraction of active incoming neighbors added explicitly. The latter tests whether a GNN adds value beyond a simple graph-derived exposure feature. Reuse the cumulative RMSE, speed, final-size, milestone, and run-percentile plots. If the GNN allows adoption with no active neighbor, it has an external-arrival mechanism that the current IC/LT models lack; identify that modeling difference or ablate it.

**Definition of done:** reproducible training and validation, no future-derived inputs, a simple learned control, autonomous rollouts under a common protocol, and results across several graph samples. A flexible fit on this one week alone is not evidence of generalization.

<a id="agents"></a>
## 10. Final milestone: prompted agents as an explicit simulation
Start with **20–50 nodes and a short horizon**. Reuse the directed graph convention and permanent-adoption rule, but make the per-node decision come from a prompt. These users have no real text or profile information in the data: every persona and message you introduce is **synthetic**.

A minimal per-step interface could be:

```text
Input: synthetic persona, current adopted/not-adopted state,
       elapsed time, number/fraction of active incoming neighbors,
       short synthetic announcement shared consistently across agents
Output: {"adopt": true or false}
```

Freeze the state at the beginning of each step. Query the inactive agents from that snapshot, validate their outputs, then apply all decisions together. Otherwise whichever user is queried first can transmit within the same step, silently changing the synchronous timing used by the baselines.

Decide before running whether isolated users may adopt from public news. If allowed, that is an external source of activation and should be compared with an equivalent baseline variant. Never show an agent its future activity label or choose personas to match test outcomes.

Record the full prompt template, persona assignments, model/version, decoding settings, step duration, initial seeds, invalid-output/retry policy, token use, and cost. A random seed alone may not make a hosted model deterministic: save raw responses and replay them when debugging. Limit concurrency on the eight-core laptop; for remote models, also respect service limits and the experiment's cost budget.

Use small pilot runs to estimate **inactive agents × steps × repetitions** before scaling. Repeat stochastic simulations, compare against IC/LT on the same small graph, and plot the same metrics. Treat differences as consequences of the declared prompt/persona assumptions—not discoveries about the real anonymized users. Any substantive new graph size, time resolution, or allowed information must be shared by the comparison models.

<a id="working"></a>
## 11. Where to make changes, and how to run
Each model family has its own folder under `experiments/`. This report, its notebook, and the IC/LT implementation live together in `experiments/heuristic/`. Shared data preparation and curve evaluation remain in `diffusion/`.

```text
experiments/heuristic/
  heuristic_models.html    This report, with embedded figures
  analysis.ipynb           Settings, explanations, and plotting cells
  models.py                IC/LT update rules and repeated simulations
  experiment.py            IC/LT parameter search and experiment orchestration
  run.py                   Rebuild this experiment's report
  benchmark/               Preserved pre-GNN metrics, curves, settings, manifest
  results/                 Optional exports from the latest run (ignored by Git)
experiments/gnn/            Separate learned-model experiment (planned)
experiments/prompted_agents/ Separate prompted-agent experiment (planned)
experiments/synthesis/      Common comparison of all three families (planned)
diffusion/data.py          Shared compressed inputs, graph sample, first-RT labels
diffusion/evaluation.py    Shared curve metrics and reachability diagnostic
data/                     Original .gz files (not tracked by Git)
docs/                     Original project proposal
README.md                 Experiment index and setup
pyproject.toml + uv.lock   Shared virtual environment dependencies
```

**To rebuild:** install Python 3.12+ and uv, then run from the project root:

```sh
uv sync --locked
uv run python experiments/heuristic/run.py
```

A collaborator must download the three named `.gz` files from [SNAP](https://snap.stanford.edu/data/higgs-twitter.html) into `data/` before rebuilding. They can read this report immediately without downloading anything. There is no server, GPU, or API key required for the baseline.

The runner executes the notebook in memory and updates `experiments/heuristic/heuristic_models.html`. The notebook on disk keeps empty outputs, avoiding duplicate figures. To edit it interactively, select the root `.venv/bin/python` as the kernel. It locates the project root from any directory inside the repository. The original root command `uv run python run.py` remains a shortcut for this experiment.

**To export figures and numerical results for further analysis:**

```sh
uv run python experiments/heuristic/run.py --export
```

This creates optional PNGs, CSVs, simulation draws, and a manifest under `experiments/heuristic/results/`. Those exports are overwritten by the next export and can become stale after a normal run. The separate `benchmark/` folder preserves the pre-GNN numerical results for future comparisons; the runner never writes there. Its manifest records settings, input hashes, source hashes, and package versions. Preserve a new run in a separately named snapshot before using it as another benchmark.

Git ignores optional exports, raw datasets, `.venv/`, and `.cache/`. The shared graph cache can be deleted and will regenerate. The report, source notebook, and compact benchmark files are intentionally kept in Git. Rebuilding the report is the end-to-end check.

### The small data contract you will reuse
- **`node_ids`: `(N,)` integer IDs.** Position i maps to row/column i everywhere. Preserve this order when building tensors or joining labels; do not align arrays by raw Twitter ID.
- **`graph`: sparse `(N, N)` binary influence adjacency.** Rows are sources, columns are recipients. No self-loops. Do not densify the full network.
- **`first_hours`: `(N,)` floats.** Hours since July 1 00:00 UTC; infinity means no observed first RT. Never feed future values to a model as features.
- **`seeds`: `(N,)` booleans.** `first_hours < SEED_HOUR`. Boundaries always count activity strictly before that hour.
- **`hours`: hourly evaluation boundaries from 24 through 168 by default.** Simulation arrays have shape `(runs, len(hours))`, including the initial count at hour 24.
- **`result`: a dictionary.** `predictions` maps model names to those arrays; `observed` is the measured cumulative curve; `best`, `tuning`, and `metrics` are compact DataFrames. Add a new model's draws to the same curve format to reuse the plots; its training/evaluation logic still needs implementing.

### Three useful experiments before adding complexity
Change one setting in the notebook's first code cell at a time: **`RANDOM_SEED = 7`**, then **`21`** to probe sample dependence; **`N_NODES = 10000`** to probe boundary effects; or **`EVALUATION_RUNS = 100`** to check Monte Carlo stability. The first run reads the full sparse graph, which needs hundreds of MB of memory; unchanged samples reuse the hidden cache. Work runs in one process with one numerical-library thread by default.

For the final project, predeclare a common comparison protocol. After inspecting a held-out result, changes motivated by that result are exploratory; do not present the same period as an untouched test of those changes.

### Assumptions to retain in your write-up
One topic, one week, one static social snapshot, and a nonrepresentative sample. First RT is participation, not verified first awareness. Prior awareness, original tweets, outside neighbors, and external news are unobserved. Active states never turn off. IC initialization, LT normalization, and the mapping of model steps to clock hours are explicit choices. Run-percentile bands measure simulation randomness only. A poor baseline is a useful result to preserve when adding model flexibility.
